In [1]:
from safetensors import torch
from transformers import AutoTokenizer

## 加载

In [2]:
# 加载BERT分词器
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-chinese")

In [3]:
print(type(tokenizer))

<class 'transformers.models.bert.tokenization_bert_fast.BertTokenizerFast'>


In [4]:
print(tokenizer)

BertTokenizerFast(name_or_path='google-bert/bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [5]:
# 从本地加载分词器
tokenizer_distil = AutoTokenizer.from_pretrained("./pretrained/bert-base-chinese")

In [6]:
print(type(tokenizer_distil))

<class 'transformers.models.distilbert.tokenization_distilbert_fast.DistilBertTokenizerFast'>


In [7]:
print(tokenizer_distil)

DistilBertTokenizerFast(name_or_path='./pretrained/bert-base-chinese', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## 使用（数据预处理）

In [8]:
text = "我爱自然语言处理"

# 1. 分词
tokens = tokenizer.tokenize(text)
print(tokens)

['我', '爱', '自', '然', '语', '言', '处', '理']


In [9]:
# 2. id化（token → id）
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415]


In [10]:
# 3. id → token
decode_tokens = tokenizer.convert_ids_to_tokens(ids)
print(decode_tokens)

['我', '爱', '自', '然', '语', '言', '处', '理']


In [12]:
# 4. 编码（encode），包含分词、id化和填充的过程
ids = tokenizer.encode(text, padding='max_length', max_length=20)
print(ids)

[101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [15]:
# 5. 解码（decode）
decode_text = tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
print(decode_text)
print(decode_text.replace(" ", ""))

我 爱 自 然 语 言 处 理
我爱自然语言处理


In [16]:
# 6. __call__方法，直接得到模型输入参数
inputs = tokenizer( text, padding='max_length', max_length=20, return_tensors='pt')
print(inputs)

{'input_ids': tensor([[ 101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [17]:
inputs_pair = tokenizer(text=text, text_pair="你爱打篮球")
print(inputs_pair)

{'input_ids': [101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415, 102, 872, 4263, 2802, 5074, 4413, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [30]:
# 7. 批量化 tokenizer 处理
texts = ["我爱自然语言处理", "你爱打篮球", "我们一起成长"]
inputs = tokenizer(
    texts, 
    padding='max_length',
    truncation=True,
    max_length=12,
    return_tensors='pt')
print(inputs)

{'input_ids': tensor([[ 101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415,  102,    0,    0],
        [ 101,  872, 4263, 2802, 5074, 4413,  102,    0,    0,    0,    0,    0],
        [ 101, 2769,  812,  671, 6629, 2768, 7270,  102,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


In [20]:
print(inputs['input_ids'].shape)

torch.Size([3, 10])


## 分词器与模型配合使用

In [37]:
import torch
from transformers import AutoModel, AutoModelForSequenceClassification

In [27]:
# 1. 准备原始文本
texts = ["我爱自然语言处理", "你爱打篮球", "我们一起成长"]

In [29]:
# 2. 加载模型和分词器
model_name = "google-bert/bert-base-chinese"
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [31]:
# 3. 调用 tokenizer 方法，将原始文本转换为模型的输入
inputs = tokenizer(
    texts,
    padding=True,
    return_tensors='pt'
)
print(inputs)

{'input_ids': tensor([[ 101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415,  102],
        [ 101,  872, 4263, 2802, 5074, 4413,  102,    0,    0,    0],
        [ 101, 2769,  812,  671, 6629, 2768, 7270,  102,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}


In [35]:
# 将inputs中的每个元素，作为参数传入，前向传播推理
with torch.no_grad():
    # outputs = model(
    #     input_ids=inputs['input_ids'],
    #     token_type_ids=inputs['token_type_ids'],
    #     attention_mask=inputs['attention_mask'],
    # )
    outputs = model(**inputs)
print(outputs.last_hidden_state.shape)
print(outputs.pooler_output.shape)

torch.Size([3, 10, 768])
torch.Size([3, 768])


#### 使用带任务头的模型

In [41]:
# 创建模型
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
inputs = tokenizer(
    texts,
    padding=True,
    return_tensors='pt'
)
print(inputs)

{'input_ids': tensor([[ 101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415,  102],
        [ 101,  872, 4263, 2802, 5074, 4413,  102,    0,    0,    0],
        [ 101, 2769,  812,  671, 6629, 2768, 7270,  102,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}


In [43]:
with torch.no_grad():
    outputs = model(**inputs)
print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.8923,  0.3146,  0.7318, -0.4912, -0.8417],
        [ 0.9483,  0.3143,  0.7101, -0.1086, -0.8757],
        [ 0.8009,  0.4994,  0.4059, -0.2941, -0.7897]]), hidden_states=None, attentions=None)
